In [51]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

In [33]:
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [ ]:
# 데이터 분할(8:2)


train_set, test_set = train_test_split(housing, test_size=0.2, random_state = 42)

# 훈련용 데이터 x, y(median_house_value)를 분류
housing = train_set.drop(columns = ['median_house_value'], axis=1)
housing_labels = train_set['median_house_value'].copy()

# 수치형, 범주형 열 파악
num_attribs = list(housing.select_dtypes(np.number).columns)
cat_attribs = ['ocean_proximity']

# 파이프라인 구축
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline(steps=[
    ('impoter', SimpleImputer(strategy='median')),('scaler', StandardScaler())
])

# SimpleImputer(strategy='median') : 결측치를 해당 열의 중앙값으로 채움
# StandardScaler() : 평균=0, 분산=1로 수치 스케일 통일

# 범주형 열과 수치형 열 처리를 하나로 묶어서 
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
preprocessing = ColumnTransformer([ # 열 종류별 다른 전처리 병렬 적용
    ('num', num_pipeline, num_attribs), # 숫치형 열을 파이프라인 으로 전치리
    ('car', OneHotEncoder(), cat_attribs) 
    # OneHotEncoder() : 문자 카테고리를 0/1로 표현 (각 카테고리를 독립적인 열로 분리)
])

# 선형회귀 모델
from sklearn.linear_model import LinearRegression
pipeline = Pipeline([
    ('preprocess', preprocessing), # preprocess : 수치형, 범주형 데이터 전처리 파이프라인
    ('model', LinearRegression())
])

pipeline.fit(housing, housing_labels)

# 평가
housing_test = test_set.drop(columns = ['median_house_value'], axis=1)
housing_labels_test = test_set['median_house_value'].copy()
pipeline.score(housing_test, housing_labels_test)


0.6486352079787185

In [35]:
housing[housing.columns]

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
14196,-117.22,32.75,34.0,6001.0,1111.0,2654.0,1072.0,4.5878,NEAR OCEAN
8267,-117.03,32.69,10.0,901.0,163.0,698.0,167.0,4.6648,NEAR OCEAN
17445,-122.27,37.74,28.0,6909.0,1554.0,2974.0,1484.0,3.6875,NEAR BAY
14265,-121.82,37.25,25.0,4021.0,634.0,2178.0,650.0,5.1663,<1H OCEAN
2271,-115.98,33.32,8.0,240.0,46.0,63.0,24.0,1.4688,INLAND
...,...,...,...,...,...,...,...,...,...
11284,-122.37,37.94,49.0,969.0,229.0,599.0,195.0,1.3167,NEAR BAY
11964,-118.38,33.89,35.0,1778.0,330.0,732.0,312.0,6.5745,<1H OCEAN
5390,-119.33,36.28,16.0,2624.0,527.0,1077.0,520.0,2.1250,INLAND
860,-117.19,34.08,22.0,2467.0,555.0,1567.0,494.0,2.6536,INLAND


In [45]:
# 의사결정나무 (Decision Tree Regressor)
# CART(Classification and Regression Tree) 알고리즘
from sklearn.tree import DecisionTreeRegressor
tree_pipeline = Pipeline([
    ('preprocess', preprocessing), 
    ('model', DecisionTreeRegressor())
]) 
tree_pipeline.fit(housing, housing_labels)
tree_pipeline.score(housing_test, housing_labels_test)

0.667259584479727

In [46]:
# 랜덤 포레스트
from sklearn.ensemble import RandomForestRegressor
random_pipeline = Pipeline([
    ('preprocess', preprocessing), 
    ('model', RandomForestRegressor())
]) 
random_pipeline.fit(housing, housing_labels)
random_pipeline.score(housing_test, housing_labels_test)

0.8233876596353662

In [52]:
# 원본에서 랜덤하게 5개 데이터 추출
import random
random.seed(42)
random_5 = random.sample(range(len(housing)), 5)
print(random_5)
print( random_pipeline.predict(housing.iloc[random_5].drop('median_house_value', axis=1)))
print( housing.iloc[random_5]['median_house_value'].values)


[3648, 819, 9012, 8024, 7314]
[ 82190.    86812.   271725.   131798.   341733.01]
[ 77200. 100000. 275000. 136500. 327300.]


In [ ]:
# 하이퍼 파라미터 튜닝 - 모델  => 모델의 설정값을 여러 조합으로 실험해 가장 좋은걸 찾음
param_grid = [
    {'model__n_estimators' : [30,100,200], 'model__max_features':[2,4,6]}
]
# model__n_estimators : 랜덤 포래스트 안에 결정 트리 개수
# model__max_features : 각 트리가 고려할 최대 특성(열) 수
# 3x3 총 9가지 조합 실험
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(random_pipeline, param_grid=param_grid, cv=3) #cv : 교차검증 3 번
grid_search.fit(housing.drop('median_house_value', axis=1), housing['median_house_value'])

,estimator,Pipeline(step...Regressor())])
,param_grid,"[{'model__max_features': [2, 4, ...], 'model__n_estimators': [30, 100, ...]}]"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('car', ...)]"


In [ ]:
print(f'최고의 파라메터 : {grid_search.best_params_}') 
print(f'최고의 점수 : {grid_search.best_score_}')
best_model = grid_search.best_estimator_ # 전처리 파이프라인 + 최적 파라미터 모델이 통째로 담겨있음
predict_value = best_model.predict( housing.iloc[random_5].drop('median_house_value',axis=1) )
print(predict_value)

최고의 파라메터 : {'model__max_features': 6, 'model__n_estimators': 200}
최고의 점수 : 0.8201102598595842
[ 81733.5  97340.  267883.5 131104.5 336928.5]
